In [1]:
import os

# Твой блок путей
USER_HOME = "/home/nshevtsova"
os.environ['HF_HOME'] = f"{USER_HOME}/.cache/huggingface"
os.environ['XDG_CACHE_HOME'] = f"{USER_HOME}/.cache"
os.environ['MPLCONFIGDIR'] = f"{USER_HOME}/.config/matplotlib"


# Создаем папки, чтобы не было Permission Denied
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)


In [2]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import open_clip
import random


/opt/conda/envs/lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/lora/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
import torch
import os
from diffusers import StableDiffusion3Pipeline
from transformers import CLIPTextModelWithProjection, T5EncoderModel, CLIPTokenizer, T5TokenizerFast

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.load_lora_weights(
    "lora_output/new_promts",
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.59it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


In [4]:
METADATA = Path("/home/nshevtsova/metadata_clean.csv")
IMG_DIR = Path("/home/nshevtsova/BCN_clean")

DATASET_LORA_DIR = "/home/nshevtsova/datasets/bio_clip"

df = pd.read_csv(METADATA)


In [5]:
import random
from glob import glob

# Словарь для хранения загруженных промптов по классам
PROMPTS_CACHE = {}

def load_prompts_for_class(cls):
    """
    Загружает все промпты из .txt файлов для конкретного класса в память.
    """
    # Если промпты уже загружены, берем из кэша
    if cls in PROMPTS_CACHE:
        return PROMPTS_CACHE[cls]
    
    class_dir = os.path.join(DATASET_LORA_DIR, cls)
    if not os.path.exists(class_dir):
        print(f"Папка датасета для класса {cls} не найдена по пути: {class_dir}")
        return []

    # Ищем все .txt файлы в папке класса
    txt_files = glob(os.path.join(class_dir, "*.txt"))
    
    prompts = []
    for txt_file in txt_files:
        try:
            with open(txt_file, "r", encoding="utf-8") as f:
                # Очищаем от лишних пробелов и берем только первую строку
                prompt = f.read().strip()
                if prompt:
                    prompts.append(prompt)
        except Exception as e:
            print(f"Ошибка чтения файла {txt_file}: {e}")
            
    if not prompts:
        print(f"Не найдено промптов в папке для класса {cls}.")
        return []
    
    # Кэшируем
    PROMPTS_CACHE[cls] = prompts
    print(f"Загружено {len(prompts)} промптов для класса {cls}")
    return prompts


def generate_images(pipe, cls, n):
    """
    Генерируем синтетические изображения для класса cls, сэмплируя промпты из датасета.
    """
    # 1. Загружаем (или берем из кэша) промпты датасета
    prompts_pool = load_prompts_for_class(cls)
    
    if not prompts_pool:
        # Фолбэк (резервный вариант), если промпты не загрузились
        print(f"Критическая ошибка: нет промптов для {cls}. Использую базовый.")
        prompt_base = f"dx_{cls.lower()}, dermoscopy image"
    else:
        print(f"Начинаю генерацию {n} картинок для {cls}, сэмплируя из {len(prompts_pool)} промптов...")

    negative = "clinical photo, patient skin, ruler, text, watermark, low quality, blurry"
    images = []
    used_prompts = []

    # Не забываем про CPU Offload, если карта Tesla V100 на 16 ГБ переполняется
    # pipe.enable_model_cpu_offload() # Если вдруг ОФМ

    for i in range(n):
        # 2. Выбираем СЛУЧАЙНЫЙ промпт из пула
        if prompts_pool:
            current_prompt = random.choice(prompts_pool)
        else:
            current_prompt = prompt_base

        with torch.no_grad(), torch.amp.autocast('cuda'):
            img = pipe(
                current_prompt,
                negative_prompt=negative,
                num_images_per_prompt=1,
                # Эти параметры мы обсуждали: они важны для качества LoRA
                guidance_scale=8,        
                num_inference_steps=30,    
                cross_attention_kwargs={"scale": 0.7}
            ).images[0]

        images.append(img)
        used_prompts.append(current_prompt)

    return images, used_prompts

In [6]:
def load_real_images(df, image_dir, cls, n=5):

    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return []

    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=42)

    images = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))

    print(f"Loaded images → real: {len(images)}")

    return images

In [7]:
fid = FrechetInceptionDistance(feature=2048).to("cpu")

def pil_to_uint8_tensor(img):
    # PIL → uint8 tensor [3,H,W]
    x = np.array(img, dtype=np.uint8)
    x = torch.from_numpy(x).permute(2, 0, 1)  # HWC → CHW
    return x

def compute_fid(real_imgs, fake_imgs):
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    for img in real_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=True)

    for img in fake_imgs:
        x = pil_to_uint8_tensor(img).unsqueeze(0).to(device)
        fid_metric.update(x, real=False)

    fid_value = fid_metric.compute().item()
    return fid_value

In [8]:
import torchvision.models as models

device = torch.device("cpu")

inception = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1,
    transform_input=False
).to("cpu").eval()

inception.fc = torch.nn.Identity()

def extract_features(images):
    feats = []
    preprocess = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor()
    ])

    with torch.no_grad():
        for img in images:
            x = preprocess(img).unsqueeze(0)
            f = inception(x)[0].detach().cpu().numpy()
            feats.append(f)

    return np.vstack(feats)


In [9]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_features(real_imgs)
    fake_feats = extract_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [10]:
os.environ['HF_TOKEN'] = "" 

In [11]:
from open_clip import create_model_from_pretrained, get_tokenizer

MY_CACHE_DIR = os.path.join(USER_HOME, ".cache", "clip")
os.makedirs(MY_CACHE_DIR, exist_ok=True)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'

print("Loading BiomedCLIP...")
model, preprocess = create_model_from_pretrained(model_id)
tokenizer = get_tokenizer(model_id)

model.to(device)
model.eval()

context_length = 256 # Специфично для BiomedCLIP

Loading BiomedCLIP...


/opt/conda/envs/lora/lib/python3.10/site-packages/open_clip/factory.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locati

In [12]:
def clip_similarity(images, texts):
    # Используем глобальные объекты из твоей ячейки инициализации
    global model, tokenizer, preprocess, device, context_length
    
    # 1. Токенизация списка текстов
    # Проверяем, что texts — это список строк
    text_tokens = tokenizer(texts, context_length=context_length).to(device)
    
    # 2. Подготовка картинок батчем (используем твое имя 'preprocess')
    # Это превратит список PIL-картинок в один тензор [100, 3, 224, 224]
    image_input = torch.stack([preprocess(img) for img in images]).to(device)
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # Получаем эмбеддинги
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_tokens)
        
        # Нормализация для расчета Cosine Similarity
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Считаем сходство пар (image[i] vs text[i])
        # .sum(dim=-1) по нормализованным векторам — это и есть косинусное сходство
        similarities = (image_features * text_features).sum(dim=-1)
        
    return similarities.mean().item()

In [ ]:
MIN_REAL = 100
N_GEN = 100

results = []

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis lesion",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}

PROMPT_MAP_NEW = {
    "AK": "<ak_skinlesion> actinic keratosis skin lesion with rough scaly surface",
    "BCC": "<bcc_skinlesion> basal cell carcinoma skin cancer with pearly border and visible blood vessels",
    "BKL": "<bkl_skinlesion> benign keratosis skin lesion with waxy stuck-on appearance", 
    "DF": "<df_skinlesion>v dermatofibroma benign skin lesion with firm structure and central white scar-like area", 
    "MEL": "<mel_skinlesion> malignant melanoma skin cancer with asymmetric shape irregular border and multiple colors", 
    "NV": "<nv_skinlesion> benign melanocytic nevus mole with symmetric round shape and uniform brown pigmentation",
    "SCC": "<scc_skinlesion> squamous cell carcinoma skin cancer with scaly surface and irregular structure",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

# удаляем все строки без класса
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

print(f"Found {len(classes)} classes: {classes}")

for cls in classes:

    print(f"\n=== Class {cls} ===")

    real_imgs = load_real_images(
        df=df,
        image_dir=IMG_DIR,
        cls=cls,
        n=N_GEN
    )

    if len(real_imgs) < 2:
        print(f"[SKIP] Not enough real images for {cls}")
        continue

    fake_imgs, fake_prompts = generate_images(
        pipe,
        cls,
        n=N_GEN
    )

    trigger = f"dx_{cls.lower()}, "
    clean_prompts = [p.replace(trigger, "") for p in fake_prompts]

    fid = compute_fid(real_imgs, fake_imgs)

    prd = compute_prd(real_imgs, fake_imgs)

    clip_sim = clip_similarity(
        fake_imgs,
        clean_prompts 
    )

    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "clip": clip_sim,
    })

    print(f"FID: {fid:.2f}")
    print(f"Precision: {prd['precision']:.3f}")
    print(f"Recall: {prd['recall']:.3f}")
    print(f"CLIP: {clip_sim:.3f}")

    del fake_imgs
    torch.cuda.empty_cache()


Found 7 classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC']

=== Class AK ===
Loaded images → real: 100
Загружено 816 промптов для класса AK
Начинаю генерацию 100 картинок для AK, сэмплируя из 816 промптов...


100%|██████████| 30/30 [00:04<00:00,  6.34it/s]


Num real: 100 Num fake: 100
FID: 239.23
Precision: 0.310
Recall: 0.700
CLIP: 0.403

=== Class BCC ===
Loaded images → real: 100
Загружено 2875 промптов для класса BCC
Начинаю генерацию 100 картинок для BCC, сэмплируя из 2875 промптов...


100%|██████████| 30/30 [00:05<00:00,  5.95it/s]


Num real: 100 Num fake: 100
FID: 236.67
Precision: 0.530
Recall: 0.340
CLIP: 0.444

=== Class BKL ===
Loaded images → real: 100
Загружено 1234 промптов для класса BKL
Начинаю генерацию 100 картинок для BKL, сэмплируя из 1234 промптов...


 47%|████▋     | 14/30 [00:02<00:02,  5.82it/s]


KeyboardInterrupt: 